In [1]:
import pandas as pd
from metagpt.tools.libs.data_preprocess import OrdinalEncode

# Load the datasets
train_df = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/08_heart_attack/train.csv')
test_df = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/08_heart_attack/test.csv')

# Copy the datasets
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Define the ordinal encoder for the 'Diet' column
diet_encoder = OrdinalEncode(features=['Diet'])

# Fit the encoder on the training data and transform both training and test data
train_df_copy = diet_encoder.fit_transform(train_df_copy)
test_df_copy = diet_encoder.transform(test_df_copy)

# Display the first few rows of the transformed datasets
train_df_copy.head(), test_df_copy.head()


2025-08-30 19:09:05.492 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


(  Patient ID  Age   Sex  ...      Continent           Hemisphere  Heart Attack Risk
 0    RCA7696   84  Male  ...  South America  Northern Hemisphere                  0
 1    RRP0438   88  Male  ...           Asia  Northern Hemisphere                  1
 2    HGB5652   90  Male  ...         Europe  Southern Hemisphere                  0
 3    MFR3315   27  Male  ...  South America  Northern Hemisphere                  0
 4    KRI1865   55  Male  ...         Africa  Southern Hemisphere                  0
 
 [5 rows x 25 columns],
   Patient ID  Age   Sex  ...      Continent           Hemisphere  Heart Attack Risk
 0    BPY8954   65  Male  ...  South America  Southern Hemisphere                  0
 1    TBS9300   77  Male  ...           Asia  Northern Hemisphere                  1
 2    MPM7379   70  Male  ...         Africa  Southern Hemisphere                  1
 3    AFI9143   47  Male  ...           Asia  Northern Hemisphere                  1
 4    FEW6960   63  Male  ...         E

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Print column information for the transformed training dataset
column_info_train = get_column_info(train_df_copy)
print("Column information for the transformed training dataset:")
print(column_info_train)

# Print column information for the transformed test dataset
column_info_test = get_column_info(test_df_copy)
print("Column information for the transformed test dataset:")
print(column_info_test)


Column information for the transformed training dataset:
{'Category': ['Patient ID', 'Sex', 'Country', 'Continent', 'Hemisphere'], 'Numeric': ['Age', 'Cholesterol', 'Heart Rate', 'Diabetes', 'Family History', 'Smoking', 'Obesity', 'Alcohol Consumption', 'Exercise Hours Per Week', 'Diet', 'Previous Heart Problems', 'Medication Use', 'Stress Level', 'Sedentary Hours Per Day', 'Income', 'BMI', 'Triglycerides', 'Physical Activity Days Per Week', 'Sleep Hours Per Day', 'Heart Attack Risk'], 'Datetime': [], 'Others': []}
Column information for the transformed test dataset:
{'Category': ['Patient ID', 'Sex', 'Country', 'Continent', 'Hemisphere'], 'Numeric': ['Age', 'Cholesterol', 'Heart Rate', 'Diabetes', 'Family History', 'Smoking', 'Obesity', 'Alcohol Consumption', 'Exercise Hours Per Week', 'Diet', 'Previous Heart Problems', 'Medication Use', 'Stress Level', 'Sedentary Hours Per Day', 'Income', 'BMI', 'Triglycerides', 'Physical Activity Days Per Week', 'Sleep Hours Per Day', 'Heart Attack 

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns
categorical_columns = ['Sex', 'Country', 'Continent', 'Hemisphere']

# Label encode categorical columns
label_encoders = {}
for column in categorical_columns:
    le = LabelEncoder()
    train_df_copy[column] = le.fit_transform(train_df_copy[column])
    test_df_copy[column] = le.transform(test_df_copy[column])
    label_encoders[column] = le

# Separate features and target
X_train = train_df_copy.drop(columns=['Patient ID', 'Heart Attack Risk'])
y_train = train_df_copy['Heart Attack Risk']
X_test = test_df_copy.drop(columns=['Patient ID', 'Heart Attack Risk'])
y_test = test_df_copy['Heart Attack Risk']

# Initialize and train the RandomForest model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Evaluate the performance of the model
roc_auc = roc_auc_score(y_test, y_pred_proba)
roc_auc


0.5033276716206653

In [4]:
# Since the model has already been trained and the data has been preprocessed,
# we can directly use the trained model to make predictions on the test set.

# Make predictions on the test set
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Evaluate the performance of the RandomForest model on the test set
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Return the ROC AUC score
roc_auc


0.5033276716206653